# conv-windowing-2d — ex1: build the 2-D conv window view via as_strided

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-windowing-2d`. Running the final beacon cell reports progress against the `CNN: 2-D conv windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 2-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-2d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-2d"
DD_SUBTOPIC = "CNN: 2-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 2-D conv windowing via `as_strided` — quick refresher

The 2-D case is the 1-D case applied to **both** spatial axes. Given `x: (B, IC, H, W)` and kernel `(KH, KW)` at stride 1, you want a view of shape `(B, IC, OH, OW, KH, KW)` where each `(KH, KW)` slice along `(OH, OW)` is one kernel-sized window.

**The stride tuple.** With input strides `(s_b, s_ic, s_h, s_w)`:

```
x.as_strided(
    size=(B, IC, OH, OW, KH, KW),
    stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
)
```

The trailing pair `(s_h, s_w)` walks *within* a window. The middle pair `(s_h, s_w)` walks *between* windows. **Same strides, different semantics** — this is the subtle teaching point of ARENA's 2-D conv: the OUTPUT spatial dims share strides with the KERNEL spatial dims, because both index into the same input rows/columns.

**Equivalence.** Contract via `einops.einsum(x_windows, weight, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')` and the result equals `F.conv2d(x, weight)` to fp tolerance.

**No data is copied** — `as_strided` only constructs a view header.

### Exercise 1 — build the 2-D conv window view via as_strided

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `as_strided` along **both** spatial axes to build the `(B, IC, OH, OW, KH, KW)` window view of a 2-D input for stride-1 convolution, and verify einsum-with-kernel matches `F.conv2d`.
> Keywords: as_strided, windowing-2d, view, conv2d-equivalence, visualization
> ```

**KCs targeted:** `windowing-2d-stride-pattern`, `windowing-2d-output-shape`

Implement `ex1_conv2d_windows(x, KH, KW)`. Given input `x: (B, IC, H, W)` and kernel sizes `KH, KW`, return the strided window view of shape `(B, IC, OH, OW, KH, KW)` where `OH = H - KH + 1`, `OW = W - KW + 1`, and each `(KH, KW)` slice along the new `(OH, OW)` axes is one stride-1 window of `x`.

**The trick.** Read `x.stride()` to get `(s_b, s_ic, s_h, s_w)`, then call:

```
x.as_strided(
    size=(B, IC, OH, OW, KH, KW),
    stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
)
```

**The teaching point.** The middle `(s_h, s_w)` advances *between* windows; the trailing `(s_h, s_w)` advances *within* a window. Same stride values, different semantic roles. Adjacent windows in `OH` overlap by `KH - 1` rows; in `OW` by `KW - 1` cols.

**Constraints.** No copy — your returned tensor must share storage with `x` (the test confirms with `.data_ptr()`).

After your view, the test contracts against a random kernel via `einops.einsum(..., 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')` and compares to `F.conv2d`.

The visualization plots one input-channel feature map alongside a sample window so you can see the windowing geometrically.

In [ ]:
def ex1_conv2d_windows(x: Tensor, KH: int, KW: int) -> Tensor:
    """Return (B, IC, OH, OW, KH, KW) window view of x for stride-1 conv2d."""
    raise NotImplementedError()


def _test_ex1():
    from torch.nn import functional as F

    # --- Shape + no-copy check ---
    rng = t.Generator().manual_seed(0)
    x = t.arange(1.0, 1 + 1 * 1 * 6 * 6).reshape(1, 1, 6, 6).contiguous()
    KH, KW = 3, 3
    win = ex1_conv2d_windows(x, KH, KW)
    OH, OW = 6 - KH + 1, 6 - KW + 1
    assert win.shape == (1, 1, OH, OW, KH, KW), (
        f'expected (1,1,{OH},{OW},{KH},{KW}), got {tuple(win.shape)}'
    )
    assert win.dtype == x.dtype
    assert win.data_ptr() == x.data_ptr(), 'must be a view (share storage with x)'

    # --- Value check at a few (oh, ow) positions ---
    for oh in range(OH):
        for ow in range(OW):
            ref = x[0, 0, oh:oh+KH, ow:ow+KW]
            got = win[0, 0, oh, ow]
            assert t.allclose(got, ref), f'window ({oh},{ow}) mismatch:\n{got}\nvs\n{ref}'

    # --- Equivalence with F.conv2d on a multi-channel input ---
    B, IC, H, W, OC = 2, 3, 14, 16, 4
    KH2, KW2 = 5, 3
    x2 = t.randn(B, IC, H, W, generator=rng)
    weight = t.randn(OC, IC, KH2, KW2, generator=rng)
    win2 = ex1_conv2d_windows(x2, KH2, KW2)
    assert win2.shape == (B, IC, H - KH2 + 1, W - KW2 + 1, KH2, KW2)
    y_manual = einops.einsum(
        win2, weight,
        'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow',
    )
    y_native = F.conv2d(x2, weight)
    assert t.allclose(y_manual, y_native, atol=1e-4), (
        'einsum(windows, weight) must equal F.conv2d to fp tolerance'
    )

    # --- Edge: KH == H, KW == W → single window of size (H, W) ---
    x3 = t.arange(25.0).reshape(1, 1, 5, 5).contiguous()
    win3 = ex1_conv2d_windows(x3, 5, 5)
    assert win3.shape == (1, 1, 1, 1, 5, 5)
    assert t.allclose(win3[0, 0, 0, 0], x3[0, 0]), 'single-window value must equal x'

    # --- Visualization: input feature map + one window highlighted ---
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    vis_x = x2[0, 0]                                          # (H, W) of batch 0, ic 0
    vis_win = win2[0, 0, 4, 6]                                # window at (oh=4, ow=6)
    axes[0].imshow(vis_x.numpy(), cmap='viridis')
    import matplotlib.patches as patches
    rect = patches.Rectangle((6 - 0.5, 4 - 0.5), KW2, KH2,
                             linewidth=2, edgecolor='red', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].set_title(f'input ic=0 with one ({KH2}x{KW2}) window highlighted')
    axes[1].imshow(vis_win.numpy(), cmap='viridis')
    axes[1].set_title('that window — view, not copy')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_conv2d_windows(x: Tensor, KH: int, KW: int) -> Tensor:
    B, IC, H, W = x.shape
    OH = H - KH + 1
    OW = W - KW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    return x.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
    )
```

**Why `(s_h, s_w)` appears twice.** The new `OH` axis means 'which window' along the height — advancing by 1 in `OH` moves the window down by 1 input row, which is `s_h` elements in storage. The new `KH` axis means 'position within a window' — also `s_h`. Same stride, different roles. Likewise for `OW` and `KW`.

**The shape size.** A `(B, IC, OH, OW, KH, KW)` tensor *looks* like it should occupy `B * IC * OH * OW * KH * KW` elements of memory — but it doesn't, because `as_strided` doesn't copy. The windows alias each other; total storage stays at `B * IC * H * W`. This is why ARENA's 'from-scratch conv' is fast.

**For strided conv (stride > 1).** Multiply the OH/OW strides by the conv stride: `stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w)`. Compute `OH = (H - KH) // SH + 1`, etc. The KH/KW pair is unchanged because we always read every position within a window.

**For padding.** Pre-pad the input with zeros (see the `conv-padding-zero` drill), then window the padded tensor. Composing the two drills gives the full ARENA conv2d.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()